# Context Formatting & Soure Attribution
- RAG에서 검색 결과를 단순히 이어 붙이면 LLM이 문서의 출처나 구분을 파악하기 어렵다.
- 검색된 Document를 더 읽기 좋은 context로 구성하고, 답변에 사용한 문서 ID를 표시하도록 만든다.

In [1]:
from dotenv import load_dotenv
load_dotenv()

LLM_MODEL = 'gpt-4.1-mini'

## 실습용 문서

In [2]:
from langchain_core.documents import Document

DOCS = [
    Document(
        page_content='RAG는 검색 단계와 생성 단계로 구성된다. 검색 단계에서는 질문과 관련 있는 문서를 찾고, 생성 단계에서는 검색된 문서를 바탕으로 답변을 만든다.',
        metadata={'doc_id': 'S01', 'title': 'RAG 처리 흐름', 'source': 'rag_intro.md'}
    ),
    Document(
        page_content='검색된 문서를 context로 넣을 때는 문서 ID, 제목, 출처 같은 metadata를 함께 제공하면 답변의 근거를 추적하기 쉽다.',
        metadata={'doc_id': 'S02', 'title': 'Context 구성', 'source': 'context_formatting.md'}
    ),
    Document(
        page_content='출처 표시는 사용자가 답변의 근거 문서를 확인할 수 있게 해 RAG 시스템의 신뢰성을 높인다.',
        metadata={'doc_id': 'S03', 'title': '출처 표시', 'source': 'source_attribution.md'}
    ),
]

def fake_retriever(query: str, k: int = 3):
    return DOCS[:k]

## 단순 context 전달과 metadata 포함 context 전달 비교

In [3]:
def format_docs_simple(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

def format_docs_with_metadata(docs):
    return '\n\n'.join(
        f"""
[{doc.metadata['doc_id']}]
제목 : {doc.metadata['title']}
출처 : {doc.metadata['source']}
내용 : {doc.page_content}
"""
for doc in docs
    )

In [4]:
docs = fake_retriever("RAG 답변에 출처를 표시해야 하는 이유는?")

print(format_docs_simple(docs))

print(format_docs_with_metadata(docs))

RAG는 검색 단계와 생성 단계로 구성된다. 검색 단계에서는 질문과 관련 있는 문서를 찾고, 생성 단계에서는 검색된 문서를 바탕으로 답변을 만든다.

검색된 문서를 context로 넣을 때는 문서 ID, 제목, 출처 같은 metadata를 함께 제공하면 답변의 근거를 추적하기 쉽다.

출처 표시는 사용자가 답변의 근거 문서를 확인할 수 있게 해 RAG 시스템의 신뢰성을 높인다.

[S01]
제목 : RAG 처리 흐름
출처 : rag_intro.md
내용 : RAG는 검색 단계와 생성 단계로 구성된다. 검색 단계에서는 질문과 관련 있는 문서를 찾고, 생성 단계에서는 검색된 문서를 바탕으로 답변을 만든다.



[S02]
제목 : Context 구성
출처 : context_formatting.md
내용 : 검색된 문서를 context로 넣을 때는 문서 ID, 제목, 출처 같은 metadata를 함께 제공하면 답변의 근거를 추적하기 쉽다.



[S03]
제목 : 출처 표시
출처 : source_attribution.md
내용 : 출처 표시는 사용자가 답변의 근거 문서를 확인할 수 있게 해 RAG 시스템의 신뢰성을 높인다.



## 출처 표시를 요구하는 prompt llm 동작

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model=LLM_MODEL,temperature=0)
output_parser = StrOutputParser()

prompt = PromptTemplate.from_template('''
당신은 검색된 문서를 바탕으로 답변하는 RAG assistant입니다.

규칙:
1. [Context]에 있는 내용만 근거로 사용하세요.
2. 질문에 직접 답하는 내용만 작성하세요.
3. 문서에 없는 내용은 추측하지 마세요.
4. 질문에 답할 근거가 부족하면 "제공된 문서만으로는 답변할 수 없습니다."라고 답하세요.
5. 답변에 사용한 문서 ID를 문장 끝에 표시하세요. 예 : [S01]
6. 답변 마지막에 "참고 문서" 항목을 작성하세요. 
                                            
[Context]
{context}
                                            
[Question]
{question}
                                            
[Answer]
''')

In [6]:
chain = prompt | llm | output_parser

question = "RAG 답변에서 출처를 표시하면 어떤 장점이 있나요?"
context1 = format_docs_simple(docs)
context2 = format_docs_with_metadata(docs)

print(chain.invoke({'context':context1,'question':question}))
print(chain.invoke({'context':context2,'question':question}))

RAG 답변에서 출처를 표시하면 사용자가 답변의 근거 문서를 확인할 수 있어 RAG 시스템의 신뢰성을 높이는 장점이 있습니다. [Context] 

참고 문서: [Context]
출처를 표시하면 사용자가 답변의 근거 문서를 확인할 수 있어 RAG 시스템의 신뢰성을 높이는 장점이 있습니다[ S03].

참고 문서: S03
